# Explainable Cyber Threat Defense Workbench

## Stage 1 — authentication data, past-only features, and transparent rules

This notebook follows the research flow of the [Explainable AML Transaction Monitoring](https://github.com/codewithdaniel1/Explainable-AML-Transaction-Monitoring) project. It begins with the LANL authentication stream and exact known red-team event matches. Later stages will add anomaly detection, trained classifiers, analyst investigation, and AI-assisted workflows.

**Label contract:** `known_redteam_event=1` means an exact match to LANL's published red-team file. `0` means **unlabeled**, not verified benign. A capped development run is for checking data and code, not final detection metrics. See [the project guide](docs/PROJECT_GUIDE.md).


## 1. Load the prepared data and manifest

Run `scripts/prepare_lanl.py` first. The notebook caps its in-memory read so early exploration remains practical. Both the source cap and notebook cap are shown below.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'docs' / 'PROJECT_GUIDE.md').exists():
    raise RuntimeError('Run this notebook with the repository as the working directory.')

PREPARED = PROJECT_ROOT / 'data' / 'processed' / 'auth_features.csv.gz'
MANIFEST = PREPARED.with_suffix('.manifest.json')
NOTEBOOK_MAX_ROWS = 1_000_000

if not PREPARED.exists() or not MANIFEST.exists():
    raise FileNotFoundError('Prepare data first: see the README command for scripts/prepare_lanl.py')
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
df = pd.read_csv(PREPARED, nrows=NOTEBOOK_MAX_ROWS, low_memory=False)
partial_read = len(df) < manifest['rows_written']
print(f"Rows loaded: {len(df):,} / {manifest['rows_written']:,}")
print(f"Known red-team keys matched in prepared source: {manifest['redteam_keys_matched']:,} / {manifest['redteam_keys_total']:,}")
print(f"Source capped: {manifest['is_capped_run']}; notebook read capped: {partial_read}")
print('Other events are unlabeled, not confirmed benign.')


## 2. Check schema, labels, and chronological coverage

The exact-key match count should be reviewed before any model comparison. A short sample may have no known red-team event at all.


In [ ]:
required = {
    'time', 'day', 'source_user', 'source_computer', 'destination_computer',
    'result', 'user_prior_events', 'user_prior_failures',
    'user_destination_prior_events', 'source_destination_prior_events',
    'user_prior_distinct_destinations', 'destination_prior_events',
    'known_redteam_event',
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing prepared columns: {sorted(missing)}')
if not df['time'].is_monotonic_increasing:
    raise ValueError('Prepared rows are not in timestamp order.')
if not set(df['known_redteam_event'].dropna().unique()).issubset({0, 1}):
    raise ValueError('The known-red-team indicator must be 0 or 1.')

profile = pd.DataFrame({
    'measure': ['rows', 'days', 'known red-team rows', 'unique source users',
                'unique source computers', 'unique destination computers'],
    'value': [len(df), df['day'].nunique(), int(df['known_redteam_event'].sum()),
              df['source_user'].nunique(), df['source_computer'].nunique(),
              df['destination_computer'].nunique()],
})
display(profile)
display(df.groupby('day').agg(events=('time', 'size'), known_redteam=('known_redteam_event', 'sum')).head(20))


In [ ]:
daily = df.groupby('day').agg(events=('time', 'size'), known_redteam=('known_redteam_event', 'sum'))
ax = daily['events'].plot(kind='bar', figsize=(12, 4), title='Authentication events by LANL day')
ax.set_ylabel('Event count')
plt.tight_layout()
plt.show()

ax = daily['known_redteam'].plot(kind='bar', figsize=(12, 3), title='Known red-team matches by LANL day')
ax.set_ylabel('Known event count')
plt.tight_layout()
plt.show()


## 3. Preview a chronological split

This is a **preview**, not a final split decision. The final boundaries must have enough known events in each evaluation period and must be fixed before testing candidate models. The notebook does not shuffle future events into training.


In [ ]:
days = np.sort(df['day'].unique())
if len(days) < 3:
    print('Fewer than three days are loaded; a chronological train/validation/test preview is unavailable.')
else:
    train_stop = max(1, int(len(days) * 0.70))
    validation_stop = min(len(days) - 1, max(train_stop + 1, int(len(days) * 0.85)))
    periods = {
        'train': days[:train_stop],
        'validation': days[train_stop:validation_stop],
        'test': days[validation_stop:],
    }
    split_rows = []
    for name, period_days in periods.items():
        subset = df[df['day'].isin(period_days)]
        split_rows.append({
            'period': name, 'first_day': int(period_days[0]) if len(period_days) else None,
            'last_day': int(period_days[-1]) if len(period_days) else None,
            'events': len(subset), 'known_redteam': int(subset['known_redteam_event'].sum()),
        })
    display(pd.DataFrame(split_rows))
    if any(row['known_redteam'] == 0 for row in split_rows):
        print('At least one period has no known event. Do not train or report comparative model metrics on this split.')


## 4. Inspect three simple rule signals

These are illustrative, transparent hypotheses. Thresholds are fixed here for the initial data audit. They are not claimed to be operational detections. We report **alert counts** and **known-event coverage**, not unqualified precision, because other events are unlabeled.


In [ ]:
success = df['result'].str.casefold().eq('success')
remote = df['source_computer'].ne(df['destination_computer'])
rule_flags = pd.DataFrame({
    'new_user_destination_after_history': (
        success & remote & df['user_prior_events'].ge(10)
        & df['user_destination_prior_events'].eq(0)
    ),
    'new_source_destination_after_history': (
        success & remote & df['user_prior_events'].ge(10)
        & df['source_destination_prior_events'].eq(0)
    ),
    'success_after_prior_failures': (
        success & remote & df['user_prior_failures'].ge(3)
    ),
}, index=df.index)
rule_flags['any_rule'] = rule_flags.any(axis=1)
known = df['known_redteam_event'].eq(1)
known_total = int(known.sum())
rule_summary = []
for name in rule_flags.columns:
    alerts = rule_flags[name]
    known_alerted = int((alerts & known).sum())
    rule_summary.append({
        'signal': name,
        'alerts': int(alerts.sum()),
        'known_events_alerted': known_alerted,
        'known_event_recall': known_alerted / known_total if known_total else np.nan,
        'alerts_per_observed_day': alerts.sum() / df['day'].nunique(),
    })
display(pd.DataFrame(rule_summary))
if not known_total:
    print('No known red-team event is loaded. Recall is undefined in this development slice.')


In [ ]:
daily_alerts = pd.DataFrame({
    'day': df['day'], 'any_rule': rule_flags['any_rule'],
    'known_redteam': known,
}).groupby('day').sum()
display(daily_alerts.head(20))
ax = daily_alerts['any_rule'].plot(kind='bar', figsize=(12, 4), title='Rule alerts by LANL day')
ax.set_ylabel('Alerts')
plt.tight_layout()
plt.show()


## 5. Next research stages

The next implementation milestone will add an Isolation Forest and user–host novelty track, then compare their alerts against these rules. Only after auditing label coverage and fixing chronological boundaries will we train the three supervised alternatives: logistic regression, XGBoost, and an Explainable Boosting Machine. Model explanations, missed-event analysis, analyst assistance, and response simulation follow in that order. No metric in this notebook should be presented as a full-dataset result when either cap above is active.
